In [ ]:
# ============================================================
# SAC IMOBILIÁRIO - NLU COMPLETO
# ============================================================

!pip install -q gensim gradio
!python -m spacy download pt_core_news_sm -q

# ============================================================
# BLOCO 1 - IMPORTAÇÕES
# ============================================================

import re
import numpy as np
import pandas as pd
import spacy
import gensim.downloader as api
from sklearn.linear_model import LogisticRegression
import gradio as gr

print("Carregando modelo Spacy...")

nlp = spacy.load("pt_core_news_sm")

print("Carregando GloVe...")

word_vectors = api.load("glove-wiki-gigaword-50")

print("Modelos carregados!")

# ============================================================
# BLOCO 1 - DATASET
# ============================================================

dados_imobiliaria = [

    ("Quero comprar um apartamento de 3 quartos com varanda", "comprar_imovel"),
    ("Gostaria de ver casas à venda no centro da cidade", "comprar_imovel"),
    ("Qual o preço médio para compra de cobertura com piscina?", "comprar_imovel"),
    ("Procuro imóvel residencial para comprar com financiamento", "comprar_imovel"),
    ("Vocês têm sobrados à venda na zona sul?", "comprar_imovel"),

    ("Procurando kitnet para alugar perto da faculdade", "alugar_imovel"),
    ("Qual o valor do aluguel deste apartamento de 2 dormitórios?", "alugar_imovel"),
    ("Quero alugar um galpão comercial para minha empresa", "alugar_imovel"),
    ("Quais imóveis estão disponíveis para locação imediata?", "alugar_imovel"),
    ("Preciso de uma casa para alugar que aceite animais", "alugar_imovel"),

    ("O chuveiro do apartamento alugado queimou, como pedir conserto?", "suporte_manutencao"),
    ("Muro da casa está com infiltração e vazamento de água", "suporte_manutencao"),
    ("Preciso do contato do encanador para reparo na cozinha", "suporte_manutencao"),
    ("A porta da varanda quebrou, quem faz a manutenção?", "suporte_manutencao"),
    ("Vazamento no teto do banheiro precisa de reparo urgente", "suporte_manutencao"),

    ("Como faço para baixar a segunda via do boleto do aluguel?", "2via_boleto_contrato"),
    ("Não recebi o boleto deste mês para pagamento", "2via_boleto_contrato"),
    ("Preciso do informe de rendimentos e cópia do contrato", "2via_boleto_contrato"),
    ("Onde pego o boleto atualizado com o valor do condomínio?", "2via_boleto_contrato"),
    ("Quero solicitar a segunda via do recibo de pagamento", "2via_boleto_contrato")
]

df = pd.DataFrame(
    dados_imobiliaria,
    columns=["mensagem", "intencao"]
)

print(f"Dataset carregado: {len(df)} mensagens")
print(f"Intenções: {df['intencao'].nunique()}")

# ============================================================
# BLOCO 2 - PRÉ-PROCESSAMENTO
# ============================================================

def preprocessar_texto(texto):

    texto_limpo = texto.lower()

    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    doc = nlp(texto_limpo)

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_space
        and len(token.text) > 1
    ]

    return " ".join(tokens)


def extrair_sentence_embedding(texto_limpo, modelo_emb):

    palavras = texto_limpo.split()

    vetores = [
        modelo_emb[p]
        for p in palavras
        if p in modelo_emb
    ]

    if len(vetores) == 0:
        return np.zeros(modelo_emb.vector_size)

    return np.mean(vetores, axis=0)


df["mensagem_limpa"] = df["mensagem"].apply(
    preprocessar_texto
)

X_densos = np.array([
    extrair_sentence_embedding(
        texto,
        word_vectors
    )
    for texto in df["mensagem_limpa"]
])

y = df["intencao"].values

print("Vetores criados!")
print("Formato de X:", X_densos.shape)

# ============================================================
# BLOCO 3 - TREINAMENTO
# ============================================================

modelo_nlu = LogisticRegression(
    C=1.0,
    max_iter=500
)

modelo_nlu.fit(X_densos, y)

print("Modelo supervisionado treinado!")

# ============================================================
# RESPOSTAS
# ============================================================

RESPOSTAS_PADRAO = {

    "comprar_imovel":
        "Atendimento de Vendas: Ficamos felizes com seu interesse! "
        "Confira nosso catálogo de imóveis à venda.",

    "alugar_imovel":
        "Atendimento de Locação: Temos ótimas opções disponíveis! "
        "Consulte nossos imóveis para aluguel.",

    "suporte_manutencao":
        "Suporte e Manutenção: Sentimos muito pelo inconveniente. "
        "Abra um chamado para que possamos acionar nossos prestadores.",

    "2via_boleto_contrato":
        "Financeiro e Contratos: Acesse a Área do Cliente "
        "para consultar boletos e documentos."
}

# ============================================================
# BLOCO 4 - FALLBACK
# ============================================================

# ALTERAÇÃO SOLICITADA:
# Antes: 0.50
# Agora: 0.65

LIMIAR_CONFIANCA = 0.65


def processar_atendimento_sac(mensagem_usuario):

    if not mensagem_usuario or not mensagem_usuario.strip():

        return (
            "N/A",
            "0.0%",
            "Aguardando mensagem...",
            "Digite uma mensagem para iniciar o atendimento."
        )

    # Pré-processamento
    msg_limpa = preprocessar_texto(mensagem_usuario)

    # Embedding
    vetor_input = extrair_sentence_embedding(
        msg_limpa,
        word_vectors
    ).reshape(1, -1)

    # Probabilidades
    probabilidades = modelo_nlu.predict_proba(
        vetor_input
    )[0]

    idx_maior_prob = np.argmax(probabilidades)

    confianca = probabilidades[idx_maior_prob]

    intencao_detectada = modelo_nlu.classes_[idx_maior_prob]

    percentual_confianca = f"{confianca * 100:.1f}%"

    # ========================================================
    # REGRA DE DECISÃO
    # ========================================================

    if confianca >= LIMIAR_CONFIANCA:

        classificacao_status = (
            f"IDENTIFICADO ({intencao_detectada}) "
            f"| Confiança: {percentual_confianca} "
            f"| Corte mínimo: 65%"
        )

        texto_resposta = RESPOSTAS_PADRAO[
            intencao_detectada
        ]

    else:

        classificacao_status = (
            f"UNCERTAIN - FALLBACK ACIONADO "
            f"| Confiança: {percentual_confianca} "
            f"| Corte mínimo: 65%"
        )

        texto_resposta = (
            "Desculpe, não consegui compreender com clareza "
            "a sua solicitação. Estou transferindo sua conversa "
            "para um de nossos atendentes."
        )

    card_resposta = f"""
    <div style="
        background-color:#f0f4f9;
        border-left:5px solid #2b5c8f;
        padding:15px;
        border-radius:8px;
        margin-top:10px;
    ">

        <h4 style="color:#2b5c8f;">
            Resposta Automática do SAC
        </h4>

        <p style="font-size:15px;">
            {texto_resposta}
        </p>

    </div>
    """

    return (
        intencao_detectada,
        percentual_confianca,
        classificacao_status,
        card_resposta
    )


# ============================================================
# TESTE AUTOMÁTICO
# ============================================================

print("\n==============================")
print("TESTE DO MODELO")
print("==============================")

teste = processar_atendimento_sac(
    "Quero comprar um apartamento"
)

print("Intenção:", teste[0])
print("Confiança:", teste[1])
print("Status:", teste[2])

# ============================================================
# BLOCO 5 - GRADIO
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="SAC Imobiliário"
) as app:

    gr.Markdown(
        """
        # 🏠 SAC Imobiliário — ChatBot Inteligente

        Digite sua necessidade abaixo.
        """
    )

    with gr.Row():

        # ESQUERDA
        with gr.Column():

            gr.Markdown("### Mensagem do Cliente")

            input_texto = gr.Textbox(
                lines=4,
                placeholder="Ex: Preciso da segunda via do boleto...",
                label="Digite sua necessidade"
            )

            btn_processar = gr.Button(
                "Processar Mensagem",
                variant="primary"
            )

            gr.Examples(
                examples=[
                    ["Preciso de suporte para consertar vazamento."],
                    ["Quero ver apartamentos à venda."],
                    ["Quero alugar um galpão comercial."],
                    ["Preciso da segunda via do boleto."],
                    ["Vocês vendem terreno na Lua ou em Marte?"]
                ],
                inputs=input_texto
            )

        # DIREITA
        with gr.Column():

            gr.Markdown("### Painel de Diagnóstico")

            with gr.Row():

                out_intencao = gr.Textbox(
                    label="Intenção",
                    interactive=False
                )

                out_confianca = gr.Textbox(
                    label="Confiança",
                    interactive=False
                )

            out_status = gr.Textbox(
                label="Status da Decisão",
                interactive=False
            )

            out_resposta = gr.HTML(
                value="Aguardando mensagem..."
            )

    btn_processar.click(
        fn=processar_atendimento_sac,
        inputs=input_texto,
        outputs=[
            out_intencao,
            out_confianca,
            out_status,
            out_resposta
        ]
    )


# ============================================================
# EXECUTAR
# ============================================================

print("\nIniciando interface Gradio...")
print("Aguarde o link aparecer abaixo...\n")

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 95.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Carregando modelo Spacy...
Carregando GloVe...
Modelos carregados!
Dataset carregado: 20 mensagens
Intenções: 4
Vetores criados!
Formato de X: (20, 50)
Modelo supervisionado treinado!

TESTE DO MODELO
Intenção: comprar_imovel
Confiança: 99.2%
Status: IDENTIFICADO (comprar_imovel) | Confiança: 99.2% | Corte mínimo: 65%


/tmp/ipykernel_2257/271948199.py:287: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(



Iniciando interface Gradio...
Aguarde o link aparecer abaixo...

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fdbc779e270f237456.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
